# LAVEN Document Reconstruction
Use a Colab GPU runtime, then run all cells and upload the scoped **laven-reconstruction-source.zip** when prompted. FAST_TRAIN uses 480 train, 120 validation and 120 held-out fictional source documents with eight epochs per model. Use FULL_TRAIN for a larger run. Runtime depends on the assigned GPU. The original LegalShield files are never imported or changed in this notebook.

In [ ]:
%pip -q install 'pymupdf>=1.27,<2' 'matplotlib>=3.8,<4' 'safetensors>=0.4,<1' 'tqdm>=4.66,<5'
import sys, subprocess, torch
from pathlib import Path
from zipfile import ZipFile
from google.colab import files, drive
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU runtime. Select a GPU runtime for fast training.')

In [ ]:
root = Path('/content/laven-reconstruction-source')
expected = {'backend/legalshield/reconstruction.py', 'research/reconstruction.py'}
uploaded = files.upload()
archive = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
if archive is None:
    raise ValueError('Upload laven-reconstruction-source.zip.')
with ZipFile(archive) as source:
    if len(source.infolist()) != len(expected) or set(source.namelist()) != expected or any(item.file_size > 100_000 for item in source.infolist()):
        raise ValueError('Use the matching scoped reconstruction source archive.')
    for name in expected:
        path = root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(source.read(name))

In [ ]:
MODE = 'FAST_TRAIN'  # FAST_TRAIN or FULL_TRAIN
USE_DRIVE = True
RESUME = True
EPOCHS = None  # Set a positive integer to override the preset
TRAIN_DOCS = None
VALIDATION_DOCS = None
TEST_DOCS = None
OCR = False  # Set True only when Tesseract is installed in this runtime.
BATCH_SIZE = 32
if USE_DRIVE:
    drive.mount('/content/drive')
    output = Path('/content/drive/MyDrive/LAVEN/reconstruction') / MODE.lower()
else:
    output = Path('/content/laven-reconstruction') / MODE.lower()
mode = {'FAST_TRAIN': 'fast', 'FULL_TRAIN': 'full'}[MODE]
cmd = [sys.executable, '-u', '-m', 'research.reconstruction', '--mode', mode, '--out', str(output), '--batch-size', str(BATCH_SIZE)]
for flag, value in {'--train': TRAIN_DOCS, '--validation': VALIDATION_DOCS, '--test': TEST_DOCS, '--epochs': EPOCHS}.items():
    if value is not None:
        cmd.extend([flag, str(value)])
if RESUME:
    cmd.append('--resume')
if OCR:
    cmd.append('--ocr')
subprocess.run(cmd, cwd=root, check=True)

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.loads((output / 'metrics.json').read_text()), indent=2))
for name in ['ae-learning', 'vae-learning', 'test-reconstructions', 'test-ssim', 'vae-samples']:
    display(Image(filename=str(output / 'figures' / (name + '.png'))))
files.download(str(output / 'laven-reconstruction.zip'))